# 승인·거절·재개가 있는 업무 그래프

재발급 신청을 미리 보여 준 뒤 승인한 경우에만 확정한다. 그래프의 호출 기록과 실제 저장 건수를 따로 확인한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 패키지가 없는 커널에서는 새 코드 셀에 다음 명령을 실행한 뒤 예제로 이동하세요.

%pip install "langgraph>=1,<2"

## 잠깐 멈춘 요청을 어느 위치에서 이어 갈까?

검토가 끝날 때까지 프로세스를 붙잡는 대신 대기 상태를 체크포인트에 남긴다. 나중에 같은 thread_id와 승인 결과를 보내 재개한다.

승인 직전에 메일을 보내거나 원장을 수정하면 거절한 건에도 변경이 남을 수 있다. 승인 노드에서는 작업 내용을 보여 주고 실제 쓰기는 승인 이후 별도 노드에 둔다.

재개가 있다는 사실만으로 외부 쓰기가 한 번만 실행되는 것은 아니다. 실제 저장소에서는 요청 ID의 유일성도 보장해야 한다.

## 확정 전 멈춘 요청 열기

수업의 LangGraph로 승인 대기까지 실행한다. 여기서 사용하는 메모리 사전은 실제 저장소를 대신하는 실습용 기록이다.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class Ticket(TypedDict, total=False):
    ticket: str
    approved: bool

ledger = {}
def review(state):
    decision = interrupt({"확정할 접수":state["ticket"]})
    return {"approved":decision is True}
def save(state):
    ledger.setdefault(state["ticket"], "확정")
    return {}

builder = StateGraph(Ticket)
builder.add_node("review", review)
builder.add_node("save", save)
builder.add_edge(START, "review")
builder.add_conditional_edges("review", lambda s:"save" if s["approved"] else "end",
                              {"save":"save","end":END})
builder.add_edge("save", END)
app = builder.compile(checkpointer=InMemorySaver())
one = {"configurable":{"thread_id":"case-1"}}
pending = app.invoke({"ticket":"T9"}, one)
print(pending["__interrupt__"][0].value, ledger)

{'확정할 접수': 'T9'} {}


대기 기록에는 T9가 있고 원장은 비었다. 같은 ID의 대기 요청을 재개해야 앞의 상태에서 결정을 이어 받는다.

### 승인과 거절 뒤 실제 저장 건수 비교

대기 중인 T9를 승인하고 다른 스레드의 T10은 거절한다. 각 결정 뒤 원장의 실제 키를 확인한다.

In [2]:
two = {"configurable":{"thread_id":"case-2"}}

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>app.invoke(Command(resume=True), one)
app.invoke({&quot;ticket&quot;:&quot;T10&quot;}, two)
app.invoke(Command(resume=False), two)
print(list(ledger))</code></pre><pre>[&#x27;T9&#x27;]
</pre><p>T9만 남았다. 같은 티켓의 확정 경로를 또 실행해도 이 사전은 한 키만 유지한다. 결제·메일처럼 사전 밖에서 발생하는 부작용까지 중복 방지되는 것은 아니다.</p></details>

## 재개 후에는 노드의 어느 줄부터 실행될까?

승인 함수 맨 앞에 관찰 로그를 추가해 본다. 이 목록은 재실행을 보여 주는 실험 장치다. 실제 청구 같은 작업을 이 위치에 두어도 되는지 판단해 보자.

In [4]:
visits = []
def observe(state):
    visits.append(state["ticket"])
    return {"approved":interrupt("확정할까요?") is True}
test = StateGraph(Ticket)
test.add_node("review", observe)
test.add_edge(START, "review")
test.add_edge("review", END)
replay = test.compile(checkpointer=InMemorySaver())
demo = {"configurable":{"thread_id":"replay-example"}}
replay.invoke({"ticket":"T11"}, demo)
print(visits)

['T11']


처음 멈추기 전 T11 한 건이 기록됐다. 이제 같은 스레드에서 거절해도 이미 수행된 로그 추가는 취소되지 않는다.

### 거절한 요청에서 노드 재실행 횟수 관찰

거절로 재개한 뒤에 visits와 최종 상태를 출력한다. 기록 두 줄이 승인 두 번을 뜻하는지도 판단한다.

In [5]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>finished = replay.invoke(Command(resume=False), demo)
print(visits, finished[&quot;approved&quot;])</code></pre><pre>[&#x27;T11&#x27;, &#x27;T11&#x27;] False
</pre><p>같은 노드의 시작 부분이 두 번 실행됐고 결정은 거절 한 번이다. 외부 변경을 승인 뒤 별도 단계로 옮기는 근거다.</p></details>

## 다른 입력에 적용하기

접수 ID를 바꾸는 수정 요청과 승인 뒤 결과 전송이 끊긴 요청을 추가한다. 수업용 DB에서 확정 키와 저장 횟수를 대조하고 재시도 경로를 설계한다.

역할을 둘로 나눈 구현과 단일 에이전트 구현에 같은 요청을 넣는다. 실제 처리 건수가 같을 때 역할 분리에 드는 호출과 얻는 검토 근거를 비교한다.